# Email Data Processing

This notebook reads email files from the `data` directory and organizes them into structured data using Pandas DataFrames

In [1]:
import os
import numpy as np
import pandas as pd
import email
from bs4 import BeautifulSoup

In [4]:
contents = []
labels = []
fnames = []

for root,dirs,files in os.walk("../data/"):
    for file in files:
        fnames.append(file)
        abs_path = os.path.join(root, file)
        try:
            if file.endswith("ipynb"):
                pass
            elif "ham" in abs_path:
                contents.append(abs_path)
                labels.append(0)
            elif "spam" in abs_path:
                contents.append(abs_path)
                labels.append(1)
            else:
                print(f"something happened you didn't expect for file {abs_path}")
        except Exception as e:
            print(f"Error processing {abs_path}: {str(e)}")
            

In [80]:
def get_multipart(msg):
    payloads = msg.get_payload()
    content_out = ""
    if type(payloads) == list:
        for part in payloads:
            content_type = part.get_content_type()
            content_disposition = part.get_content_disposition()
            charset = part.get_content_charset() or 'utf-8'
            
            if content_type == 'text/plain' and content_disposition is None:
                content1 = part.get_payload(decode=True).decode(charset, errors='replace')
                content_out=content_out+content1
            elif content_type == 'text/html' and content_disposition is None:
                soup = BeautifulSoup(part.get_payload(decode=True), 'html.parser')
                content2 = soup.get_text(separator="\n", strip=True)
                content_out=content_out+content2
    else:
        soup = BeautifulSoup(msg.get_payload(decode=True), 'html.parser')
        content_out = soup.get_text(separator="\n", strip=True)
        
    return content_out

In [81]:
def get_txt_from_type(type_, msg):
    if type_ == "text/plain":
        txt = msg.get_payload()
    elif type_ == "text/html":
        soup = BeautifulSoup(msg.get_payload(decode=True), 'html.parser')
        txt = soup.get_text(separator="\n", strip=True)
    else:
        txt = get_multipart(msg)
    return txt

In [ ]:
content_out = ""
for part in payloads:
    content_type = part.get_content_type()
    content_disposition = part.get_content_disposition()
    charset = part.get_content_charset() or 'utf-8'
    
    if content_type == 'text/plain' and content_disposition is None:
        content1 = part.get_payload(decode=True).decode(charset, errors='replace')
        content_out=content_out+content1
    elif content_type == 'text/html' and content_disposition is None:
        soup = BeautifulSoup(part.get_payload(decode=True), 'html.parser')
        content2 = soup.get_text(separator="\n", strip=True)
        content_out=content_out+content2

In [113]:
%%time
i=0
ftext = []
for item in contents:
    with open(item, "r", encoding="latin-1") as f:
        msg = email.message_from_file(f)
        type_ = msg.get_content_type()
        txt = get_txt_from_type(type_, msg)
        ftext.append(txt)

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


CPU times: user 13.7 s, sys: 2.88 s, total: 16.5 s
Wall time: 57.4 s


In [96]:
df = pd.DataFrame({
    'filename':fnames,
    'contents':ftext,
    'target':labels})

In [112]:
df.to_csv("emails.csv",index=True)

In [144]:
# None English: df.contents[27]

### Check single file
- Open single file to check for content, language

In [152]:
fc = contents[41]
with open(fc, "r", encoding="latin-1") as f:
    msg = email.message_from_file(f)
    type_ = msg.get_content_type()
    print(type_)
        
    

text/plain


In [153]:
txt = get_txt_from_type(type_, msg)
txt

'\x82à\x82à\x82ª\x82Í\x82¶\x82¯\x82Ä\x82Ô\x82Ç\x82¤\x82ª\x82ä\x82ê\x82é\n\x82µ\x82¶\x82Ý\x82Æ\x82à\x82à\x82Ì\x83R\x83\x89\x83{\x83\x8c\x81[\x83V\x83\x87\x83\x93\n\x83\x8d\x83\x8a\x81[\x83^\x83r\x83f\x83I\x81i\x82c\x82u\x82c\x81j\x90ê\x96å\n\x82¢\x82Â\x82Ü\x82Å\x89c\x8bÆ\x82Å\x82«\x82é\x82©\x82í\x82©\x82è\x82Ü\x82¹\x82ñ\n\x82²\x92\x8d\x95¶\x82Í\x82¨\x91\x81\x82ß\x82É\x81I\nhttp://book-i.net/mutou\n\x8dì\x95i\x97á\n\x8f\xad\x8f\x97\x93`\x90à\x81@\x96¼\x8cÃ\x89®\x92c\x92n9\x81@\x8f\xad\x8f\x97\x82Ì\x93¹\x91\x90\n\x82È\x82Ç\x82È\x82Ç132\x8dì\x95i\x81B\x8dD\x95]\x94\xad\x94\x84\x92\x86\x81I\n(^-^)/~\x83\x8d\x83\x8a\x98F\x97\x98\x83\x80\x83g\x81[\n\n\nTo Unsubscribe: send mail to majordomo@FreeBSD.org\nwith "unsubscribe freebsd-questions" in the body of the message\n\n'

In [155]:
soup = BeautifulSoup(msg.get_payload(decode=True), 'html.parser')
txt = soup.get_text(separator="\n", strip=True)

In [156]:
print(txt)

傕傕偑偼偠偗偰傇偳偆偑備傟傞
偟偠傒偲傕傕偺僐儔儃儗乕僔儑儞
儘儕乕僞價僨僆乮俢倁俢乯愱栧
偄偮傑偱塩嬈偱偒傞偐傢偐傝傑偣傫
偛拲暥偼偍憗傔偵両
http://book-i.net/mutou
嶌昳椺
彮彈揱愢丂柤屆壆抍抧9丂彮彈偺摴憪
側偳側偳132嶌昳丅岲昡敪攧拞両
(^-^)/~儘儕楩棙儉僩乕


To Unsubscribe: send mail to majordomo@FreeBSD.org
with "unsubscribe freebsd-questions" in the body of the message
